# BTC Orderbook LSTM — run.006 (selective prediction: rare high-conviction triggers on schema-3/4 features)

**Why:** runs 001–005 established that (a) model capacity is not the constraint (run.002's 7.4M-param model lost to a dummy; XGBoost ≈ LSTM ≈ 0 on v1 features), (b) the v1 feature set carries a small diffuse 2-min drift edge (run.004: IC_h8 +0.017, t=+4.8) that is far below the fee floor and has **no tail concentration** (run.005 gate: bottom-1% scores captured the deepest drops at 0.22× random — anti-correlated). The user's goal, stated after run.005: *"I would be happy if most is 'no idea' and some good guesses so I find trade opportunities."* With a ~$30–40 round trip, a model that is weakly right everywhere is worthless, but one that is strongly right a few times a day is a business — and pooled IC measures the first thing, not the second.

**What is new here (three things at once, deliberately — this is a feature-set + objective + metric run, not an A/B):**

1. **Schema-3/4 collector data only.** The analysis window starts where `binance_live_orderbook_v3.py` output begins (auto-detected via `future_ofi_sum`; the run aborts if fewer than `MIN_ERA_DAYS` days have accumulated). ~29 new features: Cont OFI, near/wide add–cancel flow (wide band *includes* the near band — the excess is taken offline), microprice deviation, within-bar RV / mid flips, best-level depletions, depth-message rate, per-trade size tails (p90/median), liquidation count/notional/max, book walls (qty, distance, −1 = none → mapped to band edge 0.30), 200 ms burst & same-side sweep-run concentration, ETH lead–lag, long/short ratio. All normalisations are causal, time-based rollers (run.004 convention: collector restarts land off-grid, so row-count windows lie across gaps).
2. **Tradable-opportunity heads.** The frozen LSTM trunk + Huber regression heads stay (reference readout), plus **6 binary heads**: P(close path moves ≥ θ in my favour within h bars) for side ∈ {up, dn} × h ∈ {8, 40, 240}, θ = 0.2% = **2× the taker round trip**. Labels are path-based (max favourable excursion), so "1" means a take-profit order at θ would have cleared real fees with 2× margin. Weighted BCE (per-head `pos_weight` = neg/pos capped at 20). Early stopping moves to **val average-precision of the two h40 heads** (h40 = the 10-min economics horizon; the h240 hypothesis was rejected by run.004, so `H_TRADE`/importance-weighting also moves 240 → 40 — the one deliberate change to the otherwise-frozen training loop).
3. **Selective evaluation, val-calibrated.** Trigger thresholds are **quantiles of the VALIDATION probability** (0.01% / 0.1% / 1% nominal trigger rates), then applied to test — a deployable rule, not "top-k of the test set" (which peeks at the test distribution). Reported per fold and pooled: triggers/day, path-label hit rate vs base rate (lift), **net endpoint bps after a real taker round trip**, day-cluster bootstrap 95% CIs. Variants: seed-ensemble mean (primary), **seed-unanimity abstention** (all 3 seeds must individually clear their own val threshold — realized trigger rate drops below nominal, that is the point), and a **regression-score-threshold baseline** (does the new head beat simply thresholding the frozen run.004-style score? If not, the head added nothing).

**Pre-registered decision rule (GATE, cell 12)** — at the h40 heads, primary trigger rate 0.1%, ensemble variant, for at least one side:
- **A.** pooled mean net bps > 0 **and** day-bootstrap 95% CI excludes 0;
- **B.** pooled hit-rate lift ≥ 3× base rate **and** per-fold lift ≥ 2× in ≥ 3 of 4 folds;
- **C.** neighbouring trigger rates (0.01% and 1%) also net > 0 (direction consistency — one magic threshold is noise).

All three → the selective idea works on these features; next is threshold tuning + maker-entry simulation (run.007). Any fails → the exploratory tables are hypotheses only. Distinguish failure modes: lift ≈ 1 everywhere = features still blind to opportunities; lift good but net < 0 = θ/fee geometry wrong, revisit θ and horizons before more data.

**Caveats:**
- The data window differs from runs 003–005, so regression ICs printed here are *reference only*, not comparable numbers (per-seed noise floor from run.004 was ±0.002–0.018 on a 6-month window; this window is shorter, so expect worse).
- Net bps is **hold-to-horizon** at the endpoint close; the path-label hit rate is the TP-order upper bound. A real TP trader lands in between. No slippage model.
- Short era ⇒ wide CIs. A gate fail with a CI straddling zero is "not proven", not "disproven" — rerun unchanged when more data has accumulated.
- The 0.01% rate on ~50k val samples sits beyond quantile resolution (τ ≈ val max); expect a handful of pooled triggers there. That column exists for the consistency check, not for its own CI.
- Reminder: v2 sub-bar files were missing from `btc_data.tar.xz` in runs 003–005. Rebuild the tar with **all** of v1+v2+out3+out4 before this run.

Expected runtime on a T4: **well under run.004's** (same training loop, era-restricted data).

## Setup
1. Runtime → Change runtime type → **T4 GPU**
2. Run all cells in order


In [1]:
# ── Cell 1: Install and imports ───────────────────────────────────────────
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torch',
                'scikit-learn', 'pandas', 'numpy', 'matplotlib', 'scipy'],
               check=True)

import os, glob, gc, math, random, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import average_precision_score
from scipy.stats import spearmanr
warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.backends.cudnn.benchmark = True
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')


Device: cuda
GPU: Tesla T4
VRAM: 15.6 GB


In [2]:
# ── Cell 2: (optional) data transfer helpers — uncomment what you need ────
#!apt install sshpass -y && sshpass -p XXXXXXXXXX sftp -r -o "StrictHostKeyChecking no" scpuser@155.207.120.2:btc_data.tar.xz .
#from google.colab import drive
#drive.mount('/content/drive')
#!mv btc_data.tar.xz /content/drive/MyDrive/
#!ls /content/drive/MyDrive/ -l
#drive.flush_and_unmount()


In [3]:
from google.colab import drive
drive.mount('/content/drive')

!cd ~ && mkdir -p btc_lstm_v2
!cd ~ && tar xJf /content/drive/MyDrive/btc_data.tar.xz

drive.flush_and_unmount()


Mounted at /content/drive


In [4]:
# ── Cell 3: Config ─────────────────────────────────────────────────────────

# ── CONFIGURE THESE ───────────────────────────────────────────────────────
DATA_DIR   = '/root/btc_data'
OUTPUT_DIR = '/root/btc_lstm_run006'

WINDOW_SEC  = 15          # bar size produced by the collector
HORIZONS    = [8, 40, 240]
H_TRADE     = 40          # CHANGED 240 → 40: h240 was rejected by run.004's own
                          # rule; h40 (10 min) is the economics-floor horizon and
                          # the selective-trading target. Importance sampling,
                          # Huber weighting and the reference IC all follow it.
VOL_WINDOW  = 240         # 1h rolling window for target normalisation
TARGET_CLIP = 5.0         # clip |target| at 5 sigma

SEQ_LEN      = 64         # 16 min of context
HIDDEN_DIM   = 128        # SMALL on purpose — capacity was cleared twice (run.002/003)
NUM_LAYERS   = 2
DROPOUT      = 0.3
LR           = 1e-3
WEIGHT_DECAY = 1e-4
BATCH_SIZE   = 2048
EPOCHS       = 12
WARMUP       = 2          # linear warmup epochs before cosine decay
PATIENCE     = 5          # early stop on val AP of the two h40 opportunity heads
EPOCH_SAMPLE_CAP = 600_000  # train sequences drawn per epoch (weighted, w/ replacement)

# ── run.003 (kept): focus training on directional bars ──
SAMPLE_W_BASE = 0.25      # epoch-draw probability ∝ SAMPLE_W_BASE + |y_trade|
LOSS_W_ALPHA  = 1.0       # per-sample Huber weight = min(1 + α·|y_trade|, cap)
LOSS_W_CAP    = 4.0

# ── run.004 (kept): multi-seed evaluation ──
SEEDS = [0, 1, 2]         # per-fold; test score/prob = seed-ensemble mean
MIN_DAY_SAMPLES = 500     # a test day needs this many samples to enter daily-IC stats

N_FOLDS         = 4       # walk-forward folds over the last 60% of the window
TEST_START_FRAC = 0.40    # first 40% is never tested (training warmup)
VAL_FRAC        = 0.12    # tail of each training range used for early stopping
COVERAGE_MIN    = 0.60    # features with lower non-NaN coverage are excluded

MAKER_FEE = 0.0002        # per side (Binance USDT-M futures maker)
TAKER_FEE = 0.0005        # per side

# ── run.006: selective prediction on schema-3/4 features ──
V3_ERA_ONLY  = False       # restrict the analysis window to schema-3+ rows
V3_MARKER    = 'future_ofi_sum'   # first non-NaN value of this = era start
MIN_ERA_DAYS = 45         # abort if less schema-3+ data than this has accumulated
WALL_BAND    = 0.30       # collector WALL_BAND_PCT — wall dist −1 (=absent) maps here

H_SEL        = 40         # the selective-trading horizon (10 min)
THETA_REL    = 0.002      # tradable-move threshold: 2× taker round trip (2·2·TAKER_FEE)
POS_W_CAP    = 20.0       # BCE pos_weight = neg/pos per head, capped here
BCE_W        = 1.0        # classification loss weight vs the (frozen-form) Huber part

TRIGGER_RATES = [0.0001, 0.001, 0.01]  # nominal val-calibrated trigger rates
PRIMARY_RATE  = 0.001     # pre-registered primary: top-0.1% ≈ a few signals/day
N_BOOT        = 2000      # day-cluster bootstrap draws for net-bps CIs

DRIVE_SAVE_DIR = '/content/drive/MyDrive/btc_lstm_run006'
# ─────────────────────────────────────────────────────────────────────────

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'Data dir:   {DATA_DIR}')
print(f'Output dir: {OUTPUT_DIR}')
print(f'θ = {THETA_REL*1e4:.0f} bps  (taker round trip = {2*TAKER_FEE*1e4:.0f} bps)')


Data dir:   /root/btc_data
Output dir: /root/btc_lstm_run006
θ = 20 bps  (taker round trip = 10 bps)


In [5]:
# ── Cell 4: Feature pipeline ──────────────────────────────────────────────
SEPARATOR = ';'

COLS_NEEDED_BASE = {
    'spot_datetime', 'future_datetime', 'spot_timestamp', 'future_timestamp',
    'future_bid_close', 'future_ask_close',
    'future_bid_open', 'future_bid_max', 'future_bid_min',
    'future_bid_median', 'future_ask_median',
    'future_spread_open', 'future_spread_max',
    'future_buy_qty', 'future_sell_qty',
    'future_buy_samples', 'future_sell_samples',
    'future_buy_vwap', 'future_sell_vwap', 'future_price_diff',
    'future_bid_liq_0.0_median', 'future_ask_liq_0.0_median',
    'spot_buy_qty', 'spot_sell_qty',
    'spot_bid_close', 'spot_ask_close',
    'opt_open_interest_sample', 'opt_funding_rate_sample',
    'opt_est_funding_rate_sample', 'opt_remaining_time_sample',
    'opt_long_force_exit_qty_sum', 'opt_short_force_exit_qty_sum',
    'future_first_trade_side', 'future_last_trade_side',
    'future_largest_trade_qty', 'future_largest_trade_side',
    'future_buy_count_early', 'future_buy_count_late',
    'future_sell_count_early', 'future_sell_count_late',
    'future_buy_qty_early', 'future_buy_qty_late',
    'future_sell_qty_early', 'future_sell_qty_late',
}
COLS_NEEDED_LIQ_DEEP  = [
    'future_bid_liq_0.04_median', 'future_ask_liq_0.04_median',
    'future_bid_liq_0.05_median', 'future_ask_liq_0.05_median',
    'future_bid_liq_0.06_median', 'future_ask_liq_0.06_median',
]
COLS_NEEDED_LIQ_TOTAL = [
    'future_bid_liq_0.1_median',  'future_ask_liq_0.1_median',
    'future_bid_liq_0.2_median',  'future_ask_liq_0.2_median',
    'future_bid_liq_0.3_median',  'future_ask_liq_0.3_median',
    'future_bid_liq_0.4_median',  'future_ask_liq_0.4_median',
]
# run.006: schema-3/4 collector columns (out3/out4 files; NaN in older files)
COLS_NEEDED_V3 = [
    'future_mid_std', 'future_mid_rv', 'future_mid_flips',
    'future_micro_dev_close', 'future_micro_dev_median',
    'future_ofi_sum',
    'future_add_bid_near_sum', 'future_cancel_bid_near_sum',
    'future_add_ask_near_sum', 'future_cancel_ask_near_sum',
    'future_add_bid_wide_sum', 'future_cancel_bid_wide_sum',
    'future_add_ask_wide_sum', 'future_cancel_ask_wide_sum',
    'future_depth_msg_count', 'future_best_bid_move_count',
    'future_best_ask_move_count',
    'future_bid_depletion_count', 'future_ask_depletion_count',
    'future_buy_size_median', 'future_buy_size_p90',
    'future_sell_size_median', 'future_sell_size_p90',
    'future_bid_wall_qty_median', 'future_bid_wall_qty_max',
    'future_bid_wall_dist_median',
    'future_ask_wall_qty_median', 'future_ask_wall_qty_max',
    'future_ask_wall_dist_median',
    'future_max_batch_buy_qty', 'future_max_batch_sell_qty',
    'future_max_buy_run_qty', 'future_max_sell_run_qty',
    'opt_long_force_exit_cnt_sum', 'opt_short_force_exit_cnt_sum',
    'opt_long_force_exit_notional_sum', 'opt_short_force_exit_notional_sum',
    'opt_force_exit_notional_max',
    'opt_long_short_ratio_sample',
    'opt_eth_mid_open', 'opt_eth_mid_close',
    'schema_version',
]

# run.004: long-timescale context the 64-bar input window cannot see
NEW_FEATURES = [
    'ret_norm_1h', 'ret_norm_4h', 'ret_norm_24h',
    'ma_gap_1h', 'ma_gap_4h', 'ma_gap_24h',
    'vol_ratio_1h_24h', 'range_pos_24h',
    'basis_z_4h', 'basis_mom_1h',
    'dow_sin', 'dow_cos',
]

# run.006: engineered from the schema-3/4 columns. Episodic by nature —
# mostly quiet, occasionally scream — which is the shape a rare-trigger
# model needs (v1's continuous rolling stats are why runs 001–005 came
# out diffuse).
V3_FEATURES = [
    'ofi_z', 'flow_net_near_z', 'cancel_imbal_near',
    'pull_add_bid', 'pull_add_ask', 'flow_net_widex_z',
    'micro_dev_z', 'mid_rv_norm', 'mid_flips_norm', 'msg_rate_norm',
    'depl_imbal', 'depl_total_norm',
    'buy_tail_ratio', 'sell_tail_ratio',
    'liq_cnt_log', 'liq_imbal', 'liq_notional_log', 'liq_notional_max_log',
    'wall_imbal', 'bid_wall_dist', 'ask_wall_dist', 'wall_qty_norm',
    'batch_buy_rel', 'batch_sell_rel', 'sweep_buy_rel', 'sweep_sell_rel',
    'eth_ret_z', 'eth_lead_gap', 'lsr_z',
]

ALL_FEATURES = [
    'book_imbalance', 'flow_imbalance', 'momentum', 'book_imbal_deep',
    'flow_imbal_roll4', 'flow_imbal_roll8', 'book_imbal_roll4', 'book_imbal_roll8',
    'vwap_spread', 'liq_flag', 'stochastic', 'spread_expansion', 'sample_imbalance',
    'flow_agreement', 'oi_change', 'size_imbalance', 'liq_conc_bid', 'liq_conc_ask',
    'hour_sin', 'hour_cos', 'minute_sin', 'minute_cos',
    'near_funding', 'funding_pressure', 'vol_norm',
    'trade_side_open', 'trade_side_close', 'trade_side_momentum',
    'largest_trade_side', 'largest_trade_rel',
    'buy_accel', 'sell_accel', 'flow_accel', 'buy_count_accel', 'late_imbalance',
] + NEW_FEATURES + V3_FEATURES

# binary opportunity heads: column order is frozen here and reused everywhere
CLS_KEYS = [(h, s) for h in HORIZONS for s in ('up', 'dn')]
CLS_COLS = [f'l{s}_{h}' for h, s in CLS_KEYS]
SEL_UP_J = CLS_KEYS.index((H_SEL, 'up'))
SEL_DN_J = CLS_KEYS.index((H_SEL, 'dn'))


def load_files(files):
    all_needed = (COLS_NEEDED_BASE | set(COLS_NEEDED_LIQ_DEEP)
                  | set(COLS_NEEDED_LIQ_TOTAL) | set(COLS_NEEDED_V3))
    frames = []
    for path in files:
        try:
            hdr = pd.read_csv(path, sep=SEPARATOR, nrows=0)
            usecols = sorted(all_needed & set(hdr.columns))
            if 'spot_datetime' not in usecols:
                print(f'  SKIP {os.path.basename(path)}: missing spot_datetime')
                continue
            df = pd.read_csv(path, sep=SEPARATOR, usecols=usecols, low_memory=False)
            df = df[df['spot_datetime'] != 'spot_datetime']
            frames.append(df)
        except Exception as e:
            print(f'  SKIP {os.path.basename(path)}: {e}')
    if not frames:
        raise RuntimeError('No valid CSV files found.')
    print(f'  Loaded {len(frames)} files')
    return pd.concat(frames, ignore_index=True)


def prepare(df, horizons, vol_window):
    df['dt'] = pd.to_datetime(df['spot_datetime'], errors='coerce')
    df = df.dropna(subset=['dt']).sort_values('dt').reset_index(drop=True)
    n_before = len(df)
    df = df.drop_duplicates(subset='dt', keep='first').reset_index(drop=True)
    if len(df) < n_before:
        print(f'  Dropped {n_before - len(df)} duplicate timestamps')

    skip = {'spot_datetime', 'future_datetime', 'dt'}
    for col in df.columns:
        if col not in skip:
            df[col] = pd.to_numeric(df[col], errors='coerce')

    for col in [c for c in df.columns if c.startswith('opt_')]:
        df[col] = df[col].replace(-1, np.nan)

    if 'future_bid_close' in df.columns and 'future_ask_close' in df.columns:
        df['close_price'] = (df['future_bid_close'] + df['future_ask_close']) / 2.0
    else:
        df['close_price'] = (df['future_bid_median'] + df['future_ask_median']) / 2.0

    # Gap structure: collector restarts / dropped bars break the 15s cadence.
    gap = df['dt'].diff().dt.total_seconds()
    n_breaks = int((gap != WINDOW_SEC).sum()) - 1
    big = gap[gap > 3600]
    print(f'  Cadence breaks: {n_breaks}  (gaps >1h: {len(big)}, largest: {gap.max()/3600:.1f}h)')

    eps = 1e-9
    df['flow_imbalance'] = ((df['future_buy_qty'] - df['future_sell_qty']) /
                            (df['future_buy_qty'] + df['future_sell_qty'] + eps))
    bid0 = df['future_bid_liq_0.0_median']
    ask0 = df['future_ask_liq_0.0_median']
    df['book_imbalance'] = (bid0 - ask0) / (bid0 + ask0 + eps)
    df['momentum'] = df['future_price_diff']

    deep_col = None
    for lvl in ['0.05', '0.04', '0.06']:
        b, a = f'future_bid_liq_{lvl}_median', f'future_ask_liq_{lvl}_median'
        if b in df.columns and a in df.columns:
            deep_col = (b, a); break
    df['book_imbal_deep'] = ((df[deep_col[0]] - df[deep_col[1]]) /
                             (df[deep_col[0]] + df[deep_col[1]] + eps)) if deep_col else np.nan

    df['flow_imbal_roll4'] = df['flow_imbalance'].rolling(4, min_periods=2).mean()
    df['flow_imbal_roll8'] = df['flow_imbalance'].rolling(8, min_periods=4).mean()
    df['book_imbal_roll4'] = df['book_imbalance'].rolling(4, min_periods=2).mean()
    df['book_imbal_roll8'] = df['book_imbalance'].rolling(8, min_periods=4).mean()

    df['vwap_spread'] = (df['future_buy_vwap'] - df['future_sell_vwap']
                         if 'future_buy_vwap' in df.columns else np.nan)
    liq_long  = df.get('opt_long_force_exit_qty_sum',  pd.Series(0, index=df.index))
    liq_short = df.get('opt_short_force_exit_qty_sum', pd.Series(0, index=df.index))
    df['liq_flag'] = ((liq_long + liq_short) > 0).astype(float)

    if all(c in df.columns for c in ['future_bid_max', 'future_bid_min', 'future_bid_close']):
        rng = df['future_bid_max'] - df['future_bid_min']
        df['stochastic'] = np.where(rng > 0, (df['future_bid_close'] - df['future_bid_min']) / rng, 0.5)
    else:
        df['stochastic'] = np.nan

    df['spread_expansion'] = (df['future_spread_max'] - df['future_spread_open']
                              if all(c in df.columns for c in ['future_spread_max', 'future_spread_open'])
                              else np.nan)

    if all(c in df.columns for c in ['future_buy_samples', 'future_sell_samples']):
        bs, ss = df['future_buy_samples'], df['future_sell_samples']
        df['sample_imbalance'] = (bs - ss) / (bs + ss + eps)
    else:
        df['sample_imbalance'] = np.nan

    if all(c in df.columns for c in ['spot_buy_qty', 'spot_sell_qty']):
        spot_flow = ((df['spot_buy_qty'] - df['spot_sell_qty']) /
                     (df['spot_buy_qty'] + df['spot_sell_qty'] + eps))
        df['flow_agreement'] = df['flow_imbalance'] * spot_flow
    else:
        df['flow_agreement'] = np.nan

    df['oi_change'] = df['opt_open_interest_sample'].diff() if 'opt_open_interest_sample' in df.columns else np.nan

    if all(c in df.columns for c in ['future_buy_samples', 'future_sell_samples',
                                      'future_buy_qty', 'future_sell_qty']):
        df['size_imbalance'] = (df['future_buy_qty']  / (df['future_buy_samples']  + eps) -
                                df['future_sell_qty'] / (df['future_sell_samples'] + eps))
    else:
        df['size_imbalance'] = np.nan

    sub_cols = ['future_first_trade_side', 'future_last_trade_side',
                'future_largest_trade_qty', 'future_largest_trade_side',
                'future_buy_count_early', 'future_buy_count_late',
                'future_sell_count_early', 'future_sell_count_late',
                'future_buy_qty_early', 'future_buy_qty_late',
                'future_sell_qty_early', 'future_sell_qty_late']
    if all(c in df.columns for c in sub_cols):
        bqe, bql = df['future_buy_qty_early'],  df['future_buy_qty_late']
        sqe, sql = df['future_sell_qty_early'],  df['future_sell_qty_late']
        fts, lts = df['future_first_trade_side'], df['future_last_trade_side']
        lq,  ls  = df['future_largest_trade_qty'], df['future_largest_trade_side']
        bce, bcl = df['future_buy_count_early'], df['future_buy_count_late']
        total_vol = bqe + bql + sqe + sql
        df['trade_side_open']     = fts
        df['trade_side_close']    = lts
        df['trade_side_momentum'] = lts - fts
        df['largest_trade_side']  = ls
        df['largest_trade_rel']   = lq / (total_vol + eps)
        df['buy_accel']           = bql - bqe
        df['sell_accel']          = sql - sqe
        df['flow_accel']          = (bql - sql) - (bqe - sqe)
        df['buy_count_accel']     = bcl - bce
        df['late_imbalance']      = (bql - sql) / (bql + sql + eps)
    else:
        for c in ['trade_side_open','trade_side_close','trade_side_momentum',
                  'largest_trade_side','largest_trade_rel','buy_accel','sell_accel',
                  'flow_accel','buy_count_accel','late_imbalance']:
            df[c] = np.nan

    deep_bid = next((c for c in ['future_bid_liq_0.4_median','future_bid_liq_0.3_median',
                                  'future_bid_liq_0.2_median','future_bid_liq_0.1_median']
                     if c in df.columns), None)
    deep_ask = next((c for c in ['future_ask_liq_0.4_median','future_ask_liq_0.3_median',
                                  'future_ask_liq_0.2_median','future_ask_liq_0.1_median']
                     if c in df.columns), None)
    df['liq_conc_bid'] = (df['future_bid_liq_0.0_median'] / (df[deep_bid] + eps)
                          if deep_bid else np.nan)
    df['liq_conc_ask'] = (df['future_ask_liq_0.0_median'] / (df[deep_ask] + eps)
                          if deep_ask else np.nan)

    hour, minute = df['dt'].dt.hour, df['dt'].dt.minute
    df['hour_sin']   = np.sin(2 * np.pi * hour   / 24)
    df['hour_cos']   = np.cos(2 * np.pi * hour   / 24)
    df['minute_sin'] = np.sin(2 * np.pi * minute / 60)
    df['minute_cos'] = np.cos(2 * np.pi * minute / 60)

    secs = (hour * 3600 + minute * 60 + df['dt'].dt.second) % (8 * 3600)
    df['near_funding'] = ((8 * 3600 - secs) < 900).astype(float)
    if 'opt_funding_rate_sample' in df.columns:
        funding = df['opt_funding_rate_sample'].replace(-1, np.nan).fillna(0)
        df['funding_pressure'] = funding * df['near_funding']
    else:
        df['funding_pressure'] = np.nan

    df['volatility'] = df['close_price'].diff().abs().rolling(16, min_periods=4).mean()
    df['vol_norm']   = (df['volatility'] /
                        df['volatility'].rolling(5760, min_periods=96).mean()
                       ).replace([np.inf, -np.inf], np.nan)

    # ── run.004: long-timescale context features ─────────────────────────
    # All rolling windows are TIME-based on the datetime index (rolling('24h')),
    # not row-count based: collector restarts land on arbitrary second offsets
    # and gaps would otherwise stretch a "1h" window over more wall time.
    # Everything is causal (past data only) and vol-normalised so the scale is
    # stationary across regimes; clipped at ±8 to bound scaler-era outliers.
    D = 24 * 3600 // WINDOW_SEC              # bars per 24h at full cadence
    lp = pd.Series(np.log(df['close_price'].values), index=df['dt'])

    def past(series, delta, tol_sec=2 * WINDOW_SEC):
        # value of `series` ~delta earlier (nearest bar within tol), aligned back
        p = series.reindex(series.index - delta, method='nearest',
                           tolerance=pd.Timedelta(seconds=tol_sec))
        return pd.Series(p.values, index=series.index)

    r1 = lp.diff()
    r1[gap.values != WINDOW_SEC] = np.nan    # 1-bar return across a gap is not 1-bar
    sigma24 = r1.rolling('24h', min_periods=D // 4).std()

    for name, td, W in [('1h', pd.Timedelta(hours=1),  D // 24),
                        ('4h', pd.Timedelta(hours=4),  D // 6),
                        ('24h', pd.Timedelta(hours=24), D)]:
        unit = sigma24 * np.sqrt(W) + eps
        df[f'ret_norm_{name}'] = ((lp - past(lp, td)) / unit).clip(-8, 8).values
        ma = lp.rolling(td, min_periods=int(W * 0.75)).mean()
        df[f'ma_gap_{name}'] = ((lp - ma) / unit).clip(-8, 8).values

    vol1h = r1.rolling('1h', min_periods=D // 48).std()
    df['vol_ratio_1h_24h'] = (vol1h / (sigma24 + eps)).clip(0, 5).values

    hi24 = lp.rolling('24h', min_periods=D // 4).max()
    lo24 = lp.rolling('24h', min_periods=D // 4).min()
    df['range_pos_24h'] = ((lp - lo24) / (hi24 - lo24 + eps)).clip(0, 1).values

    if all(c in df.columns for c in ['spot_bid_close', 'spot_ask_close']):
        spot_mid = (df['spot_bid_close'] + df['spot_ask_close']) / 2.0
        basis = pd.Series(((df['close_price'] - spot_mid) / (spot_mid + eps)).values,
                          index=df['dt'])
        b_ma = basis.rolling('4h', min_periods=D // 12).mean()
        b_sd = basis.rolling('4h', min_periods=D // 12).std()
        df['basis_z_4h'] = ((basis - b_ma) / (b_sd + eps)).clip(-8, 8).values
        b_mom = basis - past(basis, pd.Timedelta(hours=1))
        b_mom_sd = b_mom.rolling('4h', min_periods=D // 12).std()
        df['basis_mom_1h'] = (b_mom / (b_mom_sd + eps)).clip(-8, 8).values
    else:
        df['basis_z_4h'] = np.nan
        df['basis_mom_1h'] = np.nan

    dow = df['dt'].dt.dayofweek
    df['dow_sin'] = np.sin(2 * np.pi * dow / 7)
    df['dow_cos'] = np.cos(2 * np.pi * dow / 7)

    # ── run.006: schema-3/4 microstructure features ──────────────────────
    # Same conventions as the run.004 block: causal time-based rollers,
    # z-scores clipped ±8, ratios clipped [0, 10]. Every group is guarded on
    # column presence so pre-v3 files simply yield NaN (the era slice and the
    # coverage guard in the next cell handle the rest).
    mp1h = D // 24                                    # 1h of bars = min_periods

    def _ts(x):
        return pd.Series(np.asarray(x, dtype=np.float64), index=df['dt'])

    def roll_z(x, win='4h', mp=mp1h):
        s = _ts(x)
        return ((s - s.rolling(win, min_periods=mp).mean()) /
                (s.rolling(win, min_periods=mp).std() + eps)).clip(-8, 8).values

    def roll_ratio(x, win='4h', mp=mp1h, hi=10.0):
        s = _ts(x)
        return (s / (s.rolling(win, min_periods=mp).mean() + eps)).clip(0, hi).values

    have = set(df.columns)

    df['ofi_z'] = roll_z(df['future_ofi_sum']) if 'future_ofi_sum' in have else np.nan

    flow_near = ['future_add_bid_near_sum', 'future_cancel_bid_near_sum',
                 'future_add_ask_near_sum', 'future_cancel_ask_near_sum']
    if set(flow_near) <= have:
        ab, cb = df[flow_near[0]], df[flow_near[1]]
        aa, ca = df[flow_near[2]], df[flow_near[3]]
        df['flow_net_near_z']   = roll_z((ab - cb) - (aa - ca))
        df['cancel_imbal_near'] = ((cb - ca) / (cb + ca + eps)).values
        # bid pulled faster than replenished = support evaporating (log-diff
        # is scale-free, so no rolling normalisation needed)
        df['pull_add_bid'] = (np.log1p(cb) - np.log1p(ab)).clip(-8, 8)
        df['pull_add_ask'] = (np.log1p(ca) - np.log1p(aa)).clip(-8, 8)
    else:
        for c in ['flow_net_near_z', 'cancel_imbal_near', 'pull_add_bid', 'pull_add_ask']:
            df[c] = np.nan

    flow_wide = ['future_add_bid_wide_sum', 'future_cancel_bid_wide_sum',
                 'future_add_ask_wide_sum', 'future_cancel_ask_wide_sum']
    if set(flow_wide) <= have and set(flow_near) <= have:
        # the wide band INCLUDES the near band by construction — subtract to
        # get the 0.05–0.25% ring only (schema-4 files; NaN in out3 hours)
        abx = (df[flow_wide[0]] - df[flow_near[0]]).clip(lower=0)
        cbx = (df[flow_wide[1]] - df[flow_near[1]]).clip(lower=0)
        aax = (df[flow_wide[2]] - df[flow_near[2]]).clip(lower=0)
        cax = (df[flow_wide[3]] - df[flow_near[3]]).clip(lower=0)
        df['flow_net_widex_z'] = roll_z((abx - cbx) - (aax - cax))
    else:
        df['flow_net_widex_z'] = np.nan

    df['micro_dev_z'] = (roll_z(df['future_micro_dev_close'])
                         if 'future_micro_dev_close' in have else np.nan)
    df['mid_rv_norm']    = roll_ratio(df['future_mid_rv'])    if 'future_mid_rv'    in have else np.nan
    df['mid_flips_norm'] = roll_ratio(df['future_mid_flips']) if 'future_mid_flips' in have else np.nan
    df['msg_rate_norm']  = (roll_ratio(df['future_depth_msg_count'])
                            if 'future_depth_msg_count' in have else np.nan)

    if {'future_bid_depletion_count', 'future_ask_depletion_count'} <= have:
        bd, ad = df['future_bid_depletion_count'], df['future_ask_depletion_count']
        df['depl_imbal']      = ((bd - ad) / (bd + ad + 1.0)).values
        df['depl_total_norm'] = roll_ratio(bd + ad)
    else:
        df['depl_imbal'] = np.nan
        df['depl_total_norm'] = np.nan

    for side in ('buy', 'sell'):
        med, p90 = f'future_{side}_size_median', f'future_{side}_size_p90'
        if {med, p90} <= have:
            # fat right tail at equal volume = informed flow; log1p of the
            # ratio is scale-free across the BTC-qty drift
            df[f'{side}_tail_ratio'] = np.log1p((df[p90] / (df[med] + eps)).clip(0, 100))
        else:
            df[f'{side}_tail_ratio'] = np.nan

    # liquidations: an absent liquidation IS zero, so cnt/notional NaNs are
    # filled with 0 — but only inside the schema-3+ era, so the coverage
    # guard still sees pre-v3 rows as missing.
    m3 = df['future_ofi_sum'].notna() if 'future_ofi_sum' in have else pd.Series(False, index=df.index)
    liq_cols = ['opt_long_force_exit_cnt_sum', 'opt_short_force_exit_cnt_sum',
                'opt_long_force_exit_notional_sum', 'opt_short_force_exit_notional_sum',
                'opt_force_exit_notional_max']
    for c in liq_cols:
        if c in have:
            df.loc[m3, c] = df.loc[m3, c].fillna(0)
    if {'opt_long_force_exit_cnt_sum', 'opt_short_force_exit_cnt_sum'} <= have:
        df['liq_cnt_log'] = np.log1p(df['opt_long_force_exit_cnt_sum'] +
                                     df['opt_short_force_exit_cnt_sum'])
    else:
        df['liq_cnt_log'] = np.nan
    lqf = df.get('opt_long_force_exit_qty_sum',  pd.Series(np.nan, index=df.index)).fillna(0)
    sqf = df.get('opt_short_force_exit_qty_sum', pd.Series(np.nan, index=df.index)).fillna(0)
    df['liq_imbal'] = ((lqf - sqf) / (lqf + sqf + eps)).values
    if {'opt_long_force_exit_notional_sum', 'opt_short_force_exit_notional_sum'} <= have:
        df['liq_notional_log'] = np.log1p(df['opt_long_force_exit_notional_sum'] +
                                          df['opt_short_force_exit_notional_sum'])
    else:
        df['liq_notional_log'] = np.nan
    df['liq_notional_max_log'] = (np.log1p(df['opt_force_exit_notional_max'])
                                  if 'opt_force_exit_notional_max' in have else np.nan)

    wall_cols = ['future_bid_wall_qty_median', 'future_ask_wall_qty_median',
                 'future_bid_wall_dist_median', 'future_ask_wall_dist_median']
    if set(wall_cols) <= have:
        bq, aq = df[wall_cols[0]], df[wall_cols[1]]
        df['wall_imbal'] = ((bq - aq) / (bq + aq + eps)).values
        # dist −1 = no level in band → map to the band edge ("wall absent" =
        # "as far away as possible"); qty 0 already carries the absence signal
        df['bid_wall_dist'] = df[wall_cols[2]].where(df[wall_cols[2]] >= 0, WALL_BAND)
        df['ask_wall_dist'] = df[wall_cols[3]].where(df[wall_cols[3]] >= 0, WALL_BAND)
        df['wall_qty_norm'] = roll_ratio(bq + aq)
    else:
        for c in ['wall_imbal', 'bid_wall_dist', 'ask_wall_dist', 'wall_qty_norm']:
            df[c] = np.nan

    if {'future_max_batch_buy_qty', 'future_max_buy_run_qty'} <= have:
        # concentration of the bar's volume in one 200ms burst / one same-side
        # sweep run — bounded [0,1] per side, no normalisation needed
        df['batch_buy_rel']  = (df['future_max_batch_buy_qty']  / (df['future_buy_qty']  + eps)).clip(0, 1)
        df['batch_sell_rel'] = (df['future_max_batch_sell_qty'] / (df['future_sell_qty'] + eps)).clip(0, 1)
        df['sweep_buy_rel']  = (df['future_max_buy_run_qty']    / (df['future_buy_qty']  + eps)).clip(0, 1)
        df['sweep_sell_rel'] = (df['future_max_sell_run_qty']   / (df['future_sell_qty'] + eps)).clip(0, 1)
    else:
        for c in ['batch_buy_rel', 'batch_sell_rel', 'sweep_buy_rel', 'sweep_sell_rel']:
            df[c] = np.nan

    if {'opt_eth_mid_open', 'opt_eth_mid_close'} <= have:
        eo, ec = df['opt_eth_mid_open'], df['opt_eth_mid_close']
        eth_ret = pd.Series(np.where((eo > 0) & (ec > 0), np.log(ec / eo), np.nan),
                            index=df['dt'])
        df['eth_ret_z'] = (eth_ret / (eth_ret.rolling('4h', min_periods=mp1h).std() + eps)
                           ).clip(-8, 8).values
        # ETH ran but BTC has not (yet): 1-min cumulative z-gap
        er4 = eth_ret.rolling(4, min_periods=4).sum()
        br4 = r1.rolling(4, min_periods=4).sum()
        er4z = er4 / (er4.rolling('4h', min_periods=mp1h).std() + eps)
        br4z = br4 / (br4.rolling('4h', min_periods=mp1h).std() + eps)
        df['eth_lead_gap'] = (er4z - br4z).clip(-8, 8).values
    else:
        df['eth_ret_z'] = np.nan
        df['eth_lead_gap'] = np.nan

    df['lsr_z'] = (roll_z(df['opt_long_short_ratio_sample'], '24h', D // 4)
                   if 'opt_long_short_ratio_sample' in have else np.nan)

    # Scattered-NaN guard: a sample needs 64 contiguous surviving rows behind
    # it and 240 ahead, so a feature with even ~2% RANDOMLY scattered NaNs
    # (an ETH bar with no data, a missing long/short-ratio sample, a 4-bar
    # rolling gap echo) would wipe out most valid samples via dropna. These
    # per-bar signals are neutral-at-0 by construction (z-scores / gaps), so
    # fill them with 0 inside the schema-3+ era; warmup and pre-era rows stay
    # NaN so the coverage guard still sees them.
    for c in ['eth_ret_z', 'eth_lead_gap', 'lsr_z']:
        bad = m3 & df[c].isna() & (df['dt'] > df.loc[m3, 'dt'].iloc[0] +
                                   pd.Timedelta('24h')) if m3.any() else None
        if bad is not None:
            df.loc[bad, c] = 0.0

    del lp, r1, sigma24, vol1h, hi24, lo24
    # ──────────────────────────────────────────────────────────────────────

    # Targets: volatility-normalised future return per horizon. A target is
    # valid only when the row h bars ahead really is h*15s ahead — no labels
    # across collector gaps. vol uses only past prices (no leakage).
    # (datetime64[s] cast is robust to the pandas 2/3 resolution change)
    dt_sec = pd.Series(df['dt'].values.astype('datetime64[s]').astype(np.int64),
                       index=df.index)
    for h in horizons:
        vol   = df['close_price'].diff(h).rolling(vol_window, min_periods=vol_window // 4).std()
        delta = df['close_price'].shift(-h) - df['close_price']
        y     = (delta / (vol + eps)).clip(-TARGET_CLIP, TARGET_CLIP)
        valid = (dt_sec.shift(-h) - dt_sec) == h * WINDOW_SEC
        y[~valid.fillna(False)] = np.nan
        df[f'y_{h}'] = y

    # ── run.006: tradable-opportunity labels ─────────────────────────────
    # lup_h = 1 if the close path within the next h bars rises ≥ THETA_REL
    # above the current close (max favourable excursion for a long entered
    # now); ldn_h mirrors it for a short. θ = 2× the taker round trip, so a
    # positive label means a take-profit at θ would have cleared real fees
    # with 2× margin. Same validity rule as the regression targets: bars can
    # never be closer than 15s, so an exact h·15s span to row i+h implies
    # every intermediate bar is present.
    cvals = df['close_price'].values.astype(np.float64)
    n_rows = len(cvals)
    for h in horizons:
        fmax = np.full(n_rows, np.nan)
        fmin = np.full(n_rows, np.nan)
        if n_rows > h:
            wv = np.lib.stride_tricks.sliding_window_view(cvals, h)  # wv[j] = cvals[j:j+h]
            fmax[:n_rows - h] = wv[1:].max(axis=1)   # windows starting at i+1
            fmin[:n_rows - h] = wv[1:].min(axis=1)
        valid = ((dt_sec.shift(-h) - dt_sec) == h * WINDOW_SEC).fillna(False).values
        up = (fmax / cvals - 1.0) >= THETA_REL
        dn = (1.0 - fmin / cvals) >= THETA_REL
        df[f'lup_{h}'] = np.where(valid, up.astype(np.float32), np.nan)
        df[f'ldn_{h}'] = np.where(valid, dn.astype(np.float32), np.nan)
    return df

print('Feature pipeline loaded.')
print(f'Binary heads: {CLS_COLS}')


Feature pipeline loaded.
Binary heads: ['lup_8', 'ldn_8', 'lup_40', 'ldn_40', 'lup_240', 'ldn_240']


In [6]:
# ── Cell 5: Load data, era slice, coverage guard, build arrays ────────────
files = sorted(glob.glob(os.path.join(DATA_DIR, '*.csv')) +
               glob.glob(os.path.join(DATA_DIR, '*.csv.gz')))
print(f'Found {len(files)} files')

raw = load_files(files)
print(f'Total rows loaded: {len(raw):,}')

df = prepare(raw, HORIZONS, VOL_WINDOW)
del raw; gc.collect()

# ── run.006: restrict the analysis window to the schema-3+ era ───────────
# The v3 features only exist where the v3/v4 collector ran; over the full
# ~13-month range they would all fail the coverage guard. Features are
# computed on the FULL frame first (so the 24h rollers are warm at the era
# boundary), then the frame is sliced.
if V3_ERA_ONLY:
    mark = (df[V3_MARKER].notna() if V3_MARKER in df.columns
            else pd.Series(False, index=df.index))
    assert mark.any(), (
        f'No schema-3 data found ({V3_MARKER} entirely missing). Either the '
        f'out3/out4 files are not in btc_data.tar.xz — rebuild the tar — or '
        f'set V3_ERA_ONLY = False to run on v1 features only.')
    v3_start = df.loc[mark, 'dt'].iloc[0]
    era_days = (df['dt'].iloc[-1] - v3_start).total_seconds() / 86400
    n_pre = int((df['dt'] < v3_start).sum())
    print(f'\nSchema-3+ era starts {v3_start}  ({era_days:.1f} days, '
          f'dropping {n_pre:,} pre-era rows)')
    assert era_days >= MIN_ERA_DAYS, (
        f'Only {era_days:.1f} days of schema-3+ data (< {MIN_ERA_DAYS}). '
        f'Let the collector accumulate more before running run.006.')
    df = df[df['dt'] >= v3_start].reset_index(drop=True)
    if 'schema_version' in df.columns:
        sv = df['schema_version']
        print('  schema mix: ' + '  '.join(
            f'v{int(v)}: {(sv == v).mean()*100:.1f}%' for v in sorted(sv.dropna().unique())))

# Coverage guard (within the analysis window): schema-4-only columns (wide
# band, walls, bursts, ETH) are NaN in out3 hours; anything below
# COVERAGE_MIN is excluded rather than letting dropna() eat the window.
features = []
for f_ in ALL_FEATURES:
    if f_ not in df.columns:
        continue
    cov = df[f_].notna().mean()
    if cov >= COVERAGE_MIN:
        features.append(f_)
    else:
        print(f'  EXCLUDED {f_:22s} coverage {cov*100:5.1f}% < {COVERAGE_MIN*100:.0f}%')
kept_v3 = [f_ for f_ in features if f_ in V3_FEATURES]
print(f'Features kept: {len(features)}/{len(ALL_FEATURES)} '
      f'(schema-3/4: {len(kept_v3)}/{len(V3_FEATURES)})')
# run.006 is built to test the schema-3/4 features, but it degrades to the
# v1 feature subset when those columns are absent (v1 collector files, or a
# btc_data.tar.xz that dropped the out3/out4 files) instead of aborting.
V3_AVAILABLE = len(kept_v3) >= 10
if not V3_AVAILABLE:
    print(f'\n*** v1-compat mode: only {len(kept_v3)}/{len(V3_FEATURES)} schema-3/4 '
          f'features present ***')
    print('    The loaded files predate the schema-3/4 collector (no future_ofi_sum, '
          'walls, ETH lead-lag, ...),')
    print('    so the selective/opportunity heads train on the v1 feature subset only. '
          'This does NOT')
    print('    test run.006\'s schema-3/4 hypothesis — it reruns the run.004/005 feature '
          'set under the')
    print('    selective objective. Feed out3/out4 files (rebuild btc_data.tar.xz) for '
          'the intended run.')

n0 = len(df)
clean = df[features + [f'y_{h}' for h in HORIZONS] + CLS_COLS + ['dt', 'close_price']] \
          .dropna(subset=features).reset_index(drop=True)
lost = n0 - len(clean)
print(f'Rows: {n0:,} → {len(clean):,} after dropna ({lost/max(n0,1)*100:.2f}% dropped)')
if lost > 0.05 * n0:
    print('\n*** WARNING: dropna removed >5% of rows — check per-feature NaN fractions: ***')
    print(df[features].isna().mean().sort_values(ascending=False).head(10))
del df; gc.collect()

X_raw  = clean[features].values.astype(np.float32)
Y_all  = np.stack([clean[f'y_{h}'].values for h in HORIZONS], axis=1).astype(np.float32)
L_all  = np.stack([clean[c].values for c in CLS_COLS], axis=1).astype(np.float32)
dt_s   = clean['dt'].values.astype('datetime64[s]').astype(np.int64)
dt_all = clean['dt'].values
close  = clean['close_price'].values.astype(np.float64)
n, F_DIM = X_raw.shape
del clean; gc.collect()

# A sample "ends" at row i: input window = rows [i-SEQ_LEN+1, i], targets and
# labels at/through i+h. Backward validity: the window must be contiguous 15s
# bars. Forward validity: row i+MAX_H must be exactly MAX_H·15s ahead IN THIS
# (post-dropna) array — dropna can remove interior rows, and the evaluation
# below indexes close[e + h] positionally, so the check must hold here, not
# just in the pre-drop frame. (Improvement over run.005, which relied on the
# pre-drop check only.)
MAX_H = max(HORIZONS)
contig = np.zeros(n, dtype=bool)
span = (SEQ_LEN - 1) * WINDOW_SEC
contig[SEQ_LEN-1:] = (dt_s[SEQ_LEN-1:] - dt_s[:n-SEQ_LEN+1]) == span
fwd_ok = np.zeros(n, dtype=bool)
fwd_ok[:n-MAX_H] = (dt_s[MAX_H:] - dt_s[:n-MAX_H]) == MAX_H * WINDOW_SEC
sample_valid = (contig & fwd_ok &
                np.isfinite(Y_all).all(axis=1) & np.isfinite(L_all).all(axis=1))
assert sample_valid.sum() > 0, 'No valid samples — check bar cadence / targets'

print(f'\nRows: {n:,}   valid samples: {sample_valid.sum():,} ({sample_valid.mean()*100:.1f}%)')
print(f'Date range: {pd.Timestamp(dt_all[0])} → {pd.Timestamp(dt_all[-1])}')
H_TRADE_IDX = HORIZONS.index(H_TRADE)
yt = Y_all[sample_valid, H_TRADE_IDX]
print(f'Trading target y_{H_TRADE}: std={yt.std():.3f}   |y|>1σ: {(np.abs(yt) > 1).mean()*100:.1f}%   '
      f'up share: {(yt > 0).mean()*100:.1f}%')
print(f'\nOpportunity-label base rates (θ = {THETA_REL*1e4:.0f} bps, valid samples):')
for j, (h, s) in enumerate(CLS_KEYS):
    print(f'  {CLS_COLS[j]:8s}  {L_all[sample_valid, j].mean()*100:6.2f}%')


Found 1679 files
  Loaded 1679 files
Total rows loaded: 1,608,951
  Cadence breaks: 10494  (gaps >1h: 11, largest: 73.3h)
  EXCLUDED trade_side_open        coverage   0.0% < 60%
  EXCLUDED trade_side_close       coverage   0.0% < 60%
  EXCLUDED trade_side_momentum    coverage   0.0% < 60%
  EXCLUDED largest_trade_side     coverage   0.0% < 60%
  EXCLUDED largest_trade_rel      coverage   0.0% < 60%
  EXCLUDED buy_accel              coverage   0.0% < 60%
  EXCLUDED sell_accel             coverage   0.0% < 60%
  EXCLUDED flow_accel             coverage   0.0% < 60%
  EXCLUDED buy_count_accel        coverage   0.0% < 60%
  EXCLUDED late_imbalance         coverage   0.0% < 60%
  EXCLUDED ofi_z                  coverage   0.0% < 60%
  EXCLUDED flow_net_near_z        coverage   0.0% < 60%
  EXCLUDED cancel_imbal_near      coverage   0.0% < 60%
  EXCLUDED pull_add_bid           coverage   0.0% < 60%
  EXCLUDED pull_add_ask           coverage   0.0% < 60%
  EXCLUDED flow_net_widex_z       cove

AssertionError: Most schema-3/4 features are missing — this run exists to test them. Check the tar contents / coverage printout before proceeding.

In [ ]:
# ── Cell 6: Walk-forward folds (expanding window, purged) ─────────────────
def ends_in(a, b):
    return np.flatnonzero(sample_valid[a:b]) + a

test_start = int(n * TEST_START_FRAC)
chunk = (n - test_start) // N_FOLDS
folds = []
for k in range(N_FOLDS):
    lo = test_start + k * chunk
    hi = n if k == N_FOLDS - 1 else lo + chunk
    boundary = lo - MAX_H                     # purge: val labels must not reach into test
    val_lo   = int(boundary * (1 - VAL_FRAC))
    folds.append({'k': k,
                  'train_hi': val_lo - MAX_H,  # purge: train labels must not reach into val
                  'val':  (val_lo, boundary),
                  'test': (lo, hi)})

print('Fold   n_train     n_val    n_test    test period')
for f_ in folds:
    t_lo, t_hi = f_['test']
    n_tr = len(ends_in(0, f_['train_hi']))
    n_va = len(ends_in(*f_['val']))
    n_te = len(ends_in(t_lo, t_hi))
    assert min(n_tr, n_va, n_te) > 0, f"fold {f_['k']} has an empty split"
    span_d = (dt_s[t_hi-1] - dt_s[t_lo]) / 86400
    warn = '   *** <7 days — treat this fold as indicative only ***' if span_d < 7 else ''
    print(f"  {f_['k']}  {n_tr:>9,}  {n_va:>8,}  {n_te:>8,}    "
          f"{pd.Timestamp(dt_all[t_lo])} → {pd.Timestamp(dt_all[t_hi-1])} "
          f"({span_d:.1f}d){warn}")


In [ ]:
# ── Cell 7: Model — frozen trunk, regression heads + opportunity heads ────
class BtcLSTMSel(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers, dropout, n_reg, n_cls):
        super().__init__()
        self.input_norm = nn.LayerNorm(input_dim)
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers,
                            batch_first=True, dropout=dropout)
        self.drop = nn.Dropout(dropout)
        self.reg_head = nn.Linear(hidden_dim, n_reg)
        self.cls_head = nn.Linear(hidden_dim, n_cls)

    def forward(self, x):
        h, _ = self.lstm(self.input_norm(x))
        z = self.drop(h[:, -1])
        return self.reg_head(z), self.cls_head(z)

_tmp = BtcLSTMSel(F_DIM, HIDDEN_DIM, NUM_LAYERS, DROPOUT, len(HORIZONS), len(CLS_KEYS))
print(f'Model parameters: {sum(p.numel() for p in _tmp.parameters()):,}')
del _tmp


In [ ]:
# ── Cell 8: Training helpers (zero-copy window gathering, multi-seed) ─────
AMP = device.type == 'cuda'

def make_scheduler(opt, warmup, total):
    def fn(ep):
        if ep < warmup:
            return (ep + 1) / warmup
        prog = (ep - warmup) / max(total - warmup, 1)
        return 0.5 * (1 + math.cos(math.pi * prog))
    return torch.optim.lr_scheduler.LambdaLR(opt, fn)

def ic(a, b):
    v = spearmanr(a, b)[0]
    return float(v) if np.isfinite(v) else 0.0

def ap(y_true, p):
    # average precision with degenerate-label guards (tiny val slices can
    # have all-0 or all-1 labels; AP is undefined there)
    y = y_true > 0.5
    if y.sum() == 0 or y.all():
        return float(y.mean())
    return float(average_precision_score(y, p))

def predict(model, view, ends, batch=4096):
    model.eval()
    regs, probs = [], []
    with torch.no_grad():
        for s in range(0, len(ends), batch):
            e = ends[s:s+batch]
            xb = torch.from_numpy(view[e - (SEQ_LEN - 1)]).to(device, non_blocking=True)
            with torch.amp.autocast('cuda', enabled=AMP):
                r, lg = model(xb)
            regs.append(r.float().cpu())
            probs.append(torch.sigmoid(lg.float()).cpu())
    return torch.cat(regs).numpy(), torch.cat(probs).numpy()

def train_one_seed(seed, view, train_ends, val_ends, samp_p, pos_w):
    torch.manual_seed(seed)
    rng = np.random.default_rng(seed)

    model = BtcLSTMSel(F_DIM, HIDDEN_DIM, NUM_LAYERS, DROPOUT,
                       len(HORIZONS), len(CLS_KEYS)).to(device)
    pw = torch.tensor(pos_w, dtype=torch.float32, device=device)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    sched = make_scheduler(opt, WARMUP, EPOCHS)
    scaler_amp = torch.amp.GradScaler('cuda', enabled=AMP)

    best_ap, best_state, best_epoch, wait = -9.0, None, 0, 0
    hist = []
    for epoch in range(1, EPOCHS + 1):
        model.train()
        # weighted draw WITH replacement (weighted w/o replacement is O(k·n)
        # in numpy); duplicates are just oversampling, which is the point
        ep_ends = rng.choice(train_ends,
                             min(len(train_ends), EPOCH_SAMPLE_CAP),
                             replace=True, p=samp_p)
        tot, m = 0.0, 0
        for s in range(0, len(ep_ends), BATCH_SIZE):
            e = ep_ends[s:s+BATCH_SIZE]
            xb = torch.from_numpy(view[e - (SEQ_LEN - 1)]).to(device, non_blocking=True)
            yb = torch.from_numpy(Y_all[e]).to(device, non_blocking=True)
            lb = torch.from_numpy(L_all[e]).to(device, non_blocking=True)
            with torch.amp.autocast('cuda', enabled=AMP):
                r, lg = model(xb)
                # regression part: frozen form from run.003/004
                per = F.smooth_l1_loss(r, yb, reduction='none').mean(dim=1)
                w = torch.clamp(1.0 + LOSS_W_ALPHA * yb[:, H_TRADE_IDX].abs(),
                                max=LOSS_W_CAP)
                huber = (w * per).sum() / w.sum()
                # run.006: opportunity heads (autocast runs BCE in fp32)
                bce = F.binary_cross_entropy_with_logits(lg, lb, pos_weight=pw)
                loss = huber + BCE_W * bce
            opt.zero_grad(set_to_none=True)
            scaler_amp.scale(loss).backward()
            scaler_amp.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler_amp.step(opt)
            scaler_amp.update()
            tot += loss.item() * len(e); m += len(e)
        sched.step()

        vr, vp = predict(model, view, val_ends)
        v_ic = [ic(vr[:, i], Y_all[val_ends, i]) for i in range(len(HORIZONS))]
        v_ap_up = ap(L_all[val_ends, SEL_UP_J], vp[:, SEL_UP_J])
        v_ap_dn = ap(L_all[val_ends, SEL_DN_J], vp[:, SEL_DN_J])
        cur = 0.5 * (v_ap_up + v_ap_dn)      # early-stop metric: the run's object
        hist.append({'train_loss': tot / m, 'val_ic': v_ic,
                     'val_ap_up': v_ap_up, 'val_ap_dn': v_ap_dn})
        if cur > best_ap:
            best_ap, best_epoch, wait = cur, epoch, 0
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            flag = '  ← best'
        else:
            wait += 1
            flag = f'  ({wait}/{PATIENCE})'
        print(f'    seed {seed}  ep {epoch:2d}  loss={tot/m:.4f}  '
              f'IC_h{H_SEL}={v_ic[H_TRADE_IDX]:+.4f}  '
              f'AP_up{H_SEL}={v_ap_up:.4f}  AP_dn{H_SEL}={v_ap_dn:.4f}{flag}')
        if wait >= PATIENCE:
            print('    early stop')
            break

    model.load_state_dict(best_state)
    return model, best_state, best_epoch, hist

def train_fold(fold):
    t_hi = fold['train_hi']
    v0, v1 = fold['val']
    s0, s1 = fold['test']

    scaler = StandardScaler().fit(X_raw[:t_hi])
    Xs = scaler.transform(X_raw).astype(np.float32)
    # (n-SEQ_LEN+1, SEQ_LEN, F) VIEW into Xs — no copy; batches materialise
    # only when fancy-indexed below. This is what keeps RAM under control.
    view = np.lib.stride_tricks.sliding_window_view(Xs, (SEQ_LEN, F_DIM))[:, 0]

    train_ends = ends_in(0, t_hi)
    val_ends   = ends_in(v0, v1)
    test_ends  = ends_in(s0, s1)

    # Importance sampling: bars with bigger vol-normalised moves are drawn
    # more often; SAMPLE_W_BASE keeps flat bars in the mix so the model still
    # learns what "nothing happening" looks like.
    samp_w = SAMPLE_W_BASE + np.abs(Y_all[train_ends, H_TRADE_IDX]).astype(np.float64)
    samp_p = samp_w / samp_w.sum()
    eff = 1.0 / np.sum(samp_p ** 2) / len(samp_p)
    print(f'  sampling tilt: effective sample fraction {eff*100:.1f}% '
          f'(100% = uniform)')

    # per-head pos_weight from TRAIN base rates only (no val/test peeking)
    pos = L_all[train_ends].mean(axis=0)
    pos_w = np.clip((1.0 - pos) / np.maximum(pos, 1e-6), 1.0, POS_W_CAP)
    print('  train base rates: ' + '  '.join(
        f'{c}={p*100:.1f}%(pw{w:.1f})' for c, p, w in zip(CLS_COLS, pos, pos_w)))

    states, best_epochs, hists = [], [], []
    val_preds, test_preds, seed_test_ic = [], [], []
    seed_val_probs, seed_test_probs = [], []
    for seed in SEEDS:
        model, best_state, best_epoch, hist = train_one_seed(
            seed, view, train_ends, val_ends, samp_p, pos_w)
        vr, vp = predict(model, view, val_ends)
        tr, tp = predict(model, view, test_ends)
        s_ic = [ic(tr[:, i], Y_all[test_ends, i]) for i in range(len(HORIZONS))]
        s_ap = [ap(L_all[test_ends, j], tp[:, j]) for j in (SEL_UP_J, SEL_DN_J)]
        print(f'    seed {seed} TEST: IC_h{H_SEL}={s_ic[H_TRADE_IDX]:+.4f}  '
              f'AP_up{H_SEL}={s_ap[0]:.4f}  AP_dn{H_SEL}={s_ap[1]:.4f}   '
              f'(best epoch {best_epoch})')
        states.append(best_state); best_epochs.append(best_epoch); hists.append(hist)
        val_preds.append(vr); test_preds.append(tr); seed_test_ic.append(s_ic)
        seed_val_probs.append(vp); seed_test_probs.append(tp)
        del model

    val_pred  = np.mean(val_preds, axis=0)          # seed-ensemble (regression)
    test_pred = np.mean(test_preds, axis=0)
    val_prob  = np.mean(seed_val_probs, axis=0)     # seed-ensemble (probability)
    test_prob = np.mean(seed_test_probs, axis=0)
    test_ic = [ic(test_pred[:, i], Y_all[test_ends, i]) for i in range(len(HORIZONS))]
    test_ap = [ap(L_all[test_ends, j], test_prob[:, j]) for j in range(len(CLS_KEYS))]
    out = {'scaler': scaler, 'states': states, 'best_epochs': best_epochs,
           'hist': hists[-1], 'val_ends': val_ends, 'val_pred': val_pred,
           'test_ends': test_ends, 'test_pred': test_pred,
           'test_ic': test_ic, 'seed_test_ic': seed_test_ic,
           'val_prob': val_prob, 'test_prob': test_prob,
           'seed_val_probs': seed_val_probs, 'seed_test_probs': seed_test_probs,
           'test_ap': test_ap}
    del Xs, view
    gc.collect()
    return out


In [ ]:
# ── Cell 9: Run walk-forward ──────────────────────────────────────────────
results = []
for fold in folds:
    lo, hi = fold['test']
    print(f"\n════ Fold {fold['k']}: test {pd.Timestamp(dt_all[lo])} → {pd.Timestamp(dt_all[hi-1])} ════")
    res = train_fold(fold)
    results.append(res)
    ics = '  '.join(f'IC_h{h}={v:+.4f}' for h, v in zip(HORIZONS, res['test_ic']))
    print(f"  ENSEMBLE TEST ({len(res['test_ends']):,} samples): {ics}")

print('\n──── Regression reference (seed-ensemble; ± = per-seed spread) ────')
print('NOTE: this window differs from runs 003–005, so these ICs are context,')
print('not comparable numbers. run.004 per-seed noise floor was ±0.002–0.018.')
hdr = '  '.join(f"{'IC_h' + str(h):>16s}" for h in HORIZONS)
print(f'Fold    n_test   {hdr}')
for f_, r in zip(folds, results):
    cells = []
    for j in range(len(HORIZONS)):
        s_ics = [r['seed_test_ic'][s][j] for s in range(len(SEEDS))]
        cells.append(f"{r['test_ic'][j]:>+8.4f} ±{np.std(s_ics):.4f}")
    print(f"  {f_['k']}   {len(r['test_ends']):>8,}  " + '  '.join(cells))

ends_pool = np.concatenate([r['test_ends'] for r in results])
pred_pool = np.concatenate([r['test_pred'] for r in results])
prob_pool = np.concatenate([r['test_prob'] for r in results])
pooled_ic = [ic(pred_pool[:, i], Y_all[ends_pool, i]) for i in range(len(HORIZONS))]
print('Pooled ensemble:   ' + '  '.join(f'{v:>+9.4f}       ' for v in pooled_ic))

print('\n──── Opportunity-head test AP (seed-ensemble) vs base rate ────')
print('AP ≈ base rate = the head ranks no better than chance; the selective')
print('tables in the next cell are what actually matters — AP is a whole-')
print('distribution summary, and this run only cares about the extreme tail.')
print(f'{"head":8s}  ' + '  '.join(f'fold {k}' for k in range(N_FOLDS)) + '   pooled  base%')
for j, c in enumerate(CLS_COLS):
    aps = '  '.join(f"{r['test_ap'][j]:.4f}" for r in results)
    pooled_ap = ap(L_all[ends_pool, j], prob_pool[:, j])
    base = L_all[ends_pool, j].mean()
    print(f'{c:8s}  {aps}   {pooled_ap:.4f}  {base*100:5.2f}')

# Backtest/pooling score: per-fold predictions divided by that fold's
# VALIDATION score std, so "σ" thresholds mean the same thing in every fold
# and the score scale is chosen without touching test data.
score_pool = np.concatenate([r['test_pred'] / (r['val_pred'].std(axis=0) + 1e-9)
                             for r in results])

def daily_ic_stats(ends, score, y):
    days = dt_all[ends].astype('datetime64[D]')
    vals = []
    for d in np.unique(days):
        m = days == d
        if m.sum() >= MIN_DAY_SAMPLES:
            vals.append(ic(score[m], y[m]))
    vals = np.array(vals)
    se = vals.std(ddof=1) / np.sqrt(len(vals))
    return vals.mean(), se, vals.mean() / se, len(vals)

print('\n──── Daily-IC t-stats (pooled ensemble; |t|>2 ≈ significant) ────')
for j, h in enumerate(HORIZONS):
    mu, se, t, k = daily_ic_stats(ends_pool, score_pool[:, j], Y_all[ends_pool, j])
    print(f'  h{h:<4d}  mean daily IC {mu:+.4f} ± {se:.4f}   t = {t:+.2f}   ({k} days)')

# ── persist scores IMMEDIATELY — run.004's scores died with the VM ────────
fold_of = np.concatenate([np.full(len(r['test_ends']), f_['k'])
                          for f_, r in zip(folds, results)])
val_ends_pool = np.concatenate([r['val_ends'] for r in results])
val_prob_pool = np.concatenate([r['val_prob'] for r in results])
val_fold_of   = np.concatenate([np.full(len(r['val_ends']), f_['k'])
                                for f_, r in zip(folds, results)])
seed_prob_pool     = np.concatenate([np.stack(r['seed_test_probs'], axis=2) for r in results])
seed_val_prob_pool = np.concatenate([np.stack(r['seed_val_probs'],  axis=2) for r in results])
scores_path = os.path.join(OUTPUT_DIR, 'run006_scores.npz')
np.savez_compressed(scores_path,
    ends=ends_pool, score=score_pool, prob=prob_pool, seed_prob=seed_prob_pool,
    labels=L_all[ends_pool], y=Y_all[ends_pool],
    horizons=np.array(HORIZONS), cls_cols=np.array(CLS_COLS),
    dt_s=dt_s[ends_pool], close=close[ends_pool], fold_of=fold_of,
    val_ends=val_ends_pool, val_prob=val_prob_pool, val_fold_of=val_fold_of,
    seed_val_prob=seed_val_prob_pool, val_dt_s=dt_s[val_ends_pool])
print(f'\nScores persisted → {scores_path}')
print('(copied to Drive in the final cell — if you plan to stop early, copy it now)')


In [ ]:
# ── Cell 10: Selective evaluation — val-calibrated triggers ───────────────
# The deployable rule: pick the trigger threshold as a quantile of the
# VALIDATION probability (never the test set — "top-k of test" quietly uses
# the test distribution to place the cut), apply it to test, and ask what a
# taker who blindly follows every trigger earns after real fees.
#
# hit%  = path-label rate among triggers (a TP order at θ would have filled:
#         the optimistic bound). net bps = endpoint move at h in the trade
#         direction minus the taker round trip (the pessimistic hold-to-
#         horizon bound). A real TP trader lands in between.
# CI    = day-cluster bootstrap of the pooled mean (triggers cluster in
#         time; treating them as independent would shrink the CI ~n-fold).

RT_BPS = 2 * TAKER_FEE * 1e4

def day_of(e):
    return dt_all[e].astype('datetime64[D]')

def net_bps(e, side, h):
    fwd = close[e + h] / close[e] - 1.0
    sgn = 1.0 if side == 'up' else -1.0
    return sgn * fwd * 1e4 - RT_BPS

def boot_ci_mean(x, days, n_boot=N_BOOT, seed=0):
    if len(x) == 0:
        return (np.nan, np.nan)
    rng = np.random.default_rng(seed)
    uds, inv = np.unique(days, return_inverse=True)
    sums = np.bincount(inv, weights=x, minlength=len(uds))
    cnts = np.bincount(inv, minlength=len(uds)).astype(np.float64)
    idx = rng.integers(0, len(uds), size=(n_boot, len(uds)))
    bs = sums[idx].sum(axis=1) / np.maximum(cnts[idx].sum(axis=1), 1.0)
    return (float(np.percentile(bs, 2.5)), float(np.percentile(bs, 97.5)))

def trigger_stats(j, rate, variant='ens'):
    # j = CLS head index; variant: 'ens' (seed-ensemble prob), 'unan' (all
    # seeds beyond their own val threshold), 'reg' (frozen regression score
    # thresholded — the "did the new head add anything" baseline).
    h, side = CLS_KEYS[j]
    ridx = HORIZONS.index(h)
    per_fold, E, NB, HT = [], [], [], []
    for f_, r in zip(folds, results):
        if variant == 'reg':
            v, t = r['val_pred'][:, ridx], r['test_pred'][:, ridx]
            trig = (t >= np.quantile(v, 1 - rate)) if side == 'up' \
                   else (t <= np.quantile(v, rate))
        else:
            tau = np.quantile(r['val_prob'][:, j], 1 - rate)
            trig = r['test_prob'][:, j] >= tau
            if variant == 'unan':
                for s in range(len(SEEDS)):
                    tau_s = np.quantile(r['seed_val_probs'][s][:, j], 1 - rate)
                    trig &= r['seed_test_probs'][s][:, j] >= tau_s
        e = r['test_ends'][trig]
        base = float(L_all[r['test_ends'], j].mean())
        nb = net_bps(e, side, h)
        n_days = max(len(np.unique(day_of(r['test_ends']))), 1)
        hit = float(L_all[e, j].mean()) if len(e) else np.nan
        per_fold.append(dict(k=f_['k'], n=len(e), per_day=len(e) / n_days,
                             hit=hit, base=base,
                             lift=(hit / base) if (len(e) and base > 0) else np.nan,
                             net=float(nb.mean()) if len(e) else np.nan))
        E.append(e); NB.append(nb); HT.append(L_all[e, j])
    e = np.concatenate(E); nb = np.concatenate(NB); ht = np.concatenate(HT)
    base_p = float(L_all[ends_pool, j].mean())
    n_days = max(len(np.unique(day_of(ends_pool))), 1)
    hit_p = float(ht.mean()) if len(e) else np.nan
    pooled = dict(n=len(e), per_day=len(e) / n_days, hit=hit_p, base=base_p,
                  lift=(hit_p / base_p) if (len(e) and base_p > 0) else np.nan,
                  net=float(nb.mean()) if len(e) else np.nan,
                  ci=boot_ci_mean(nb, day_of(e)))
    return per_fold, pooled

def _fmt(v, spec, na='     —'):
    return format(v, spec) if np.isfinite(v) else na

def show(j, rate, variant='ens', with_folds=False):
    pf, p = trigger_stats(j, rate, variant)
    h, side = CLS_KEYS[j]
    line = (f'  {side}{h:<4d} {rate*100:7.3f}%  {variant:4s}  {p["n"]:>6,}  '
            f'{p["per_day"]:>6.2f}  {_fmt(p["hit"]*100 if np.isfinite(p["hit"]) else np.nan, "6.1f")}  '
            f'{p["base"]*100:6.2f}  {_fmt(p["lift"], "5.2f")}  '
            f'{_fmt(p["net"], "+8.2f")}  [{_fmt(p["ci"][0], "+7.2f")}, {_fmt(p["ci"][1], "+7.2f")}]')
    print(line)
    if with_folds:
        for r_ in pf:
            print(f'        fold {r_["k"]}: n={r_["n"]:<6,} hit={_fmt(r_["hit"]*100 if np.isfinite(r_["hit"]) else np.nan, "5.1f")}% '
                  f'lift={_fmt(r_["lift"], "5.2f")} net={_fmt(r_["net"], "+8.2f")} bps')
    return pf, p

HDR = ('  head    rate     var       n    /day    hit%   base%   lift   net bps  [95% CI, day-boot]')

print(f'════ h{H_SEL} opportunity heads — the run.006 headline ════')
print(HDR)
for j in (SEL_UP_J, SEL_DN_J):
    for rate in TRIGGER_RATES:
        show(j, rate, 'ens', with_folds=(rate == PRIMARY_RATE))

print('\n════ Seed-unanimity abstention (all 3 seeds beyond their own val τ) ════')
print('Realized trigger rate < nominal by design — disagreement = "no idea".')
print(HDR)
for j in (SEL_UP_J, SEL_DN_J):
    for rate in TRIGGER_RATES:
        show(j, rate, 'unan')

print('\n════ Baseline: frozen regression score, same val-calibrated rates ════')
print('If the opportunity heads do not beat this, the BCE heads added nothing')
print('over thresholding the run.004-style score.')
print(HDR)
for j in (SEL_UP_J, SEL_DN_J):
    for rate in TRIGGER_RATES:
        show(j, rate, 'reg')

print(f'\n════ Exploratory: other horizons at the primary rate ({PRIMARY_RATE*100:.1f}%) ════')
print('Hypotheses for run.007, not evidence.')
print(HDR)
for j, (h, s) in enumerate(CLS_KEYS):
    if h != H_SEL:
        show(j, PRIMARY_RATE, 'ens')

# ── non-overlapping trade simulation at the primary setting ───────────────
# Per-signal stats above count overlapping triggers (bar i and i+1 both
# firing = the same move twice). This greedily takes one trade at a time,
# holds h bars, ignores triggers meanwhile — a floor on realizable activity.
def non_overlap(j, rate, variant='ens'):
    h, side = CLS_KEYS[j]
    E = []
    for f_, r in zip(folds, results):
        pf, _ = None, None
        if variant == 'reg':
            ridx = HORIZONS.index(h)
            v, t = r['val_pred'][:, ridx], r['test_pred'][:, ridx]
            trig = (t >= np.quantile(v, 1 - rate)) if side == 'up' \
                   else (t <= np.quantile(v, rate))
        else:
            tau = np.quantile(r['val_prob'][:, j], 1 - rate)
            trig = r['test_prob'][:, j] >= tau
            if variant == 'unan':
                for s in range(len(SEEDS)):
                    tau_s = np.quantile(r['seed_val_probs'][s][:, j], 1 - rate)
                    trig &= r['seed_test_probs'][s][:, j] >= tau_s
        E.append(r['test_ends'][trig])
    ends = np.concatenate(E)
    sgn = 1.0 if side == 'up' else -1.0
    usd, last_exit = [], -1
    for e in ends:                       # test ranges are ordered and disjoint
        if e < last_exit:
            continue
        usd.append(sgn * (close[e + h] - close[e]) -
                   (close[e] + close[e + h]) * TAKER_FEE)
        last_exit = e + h
    usd = np.array(usd)
    tag = f'{side}{h} @{rate*100:.1f}% {variant}'
    if len(usd) == 0:
        print(f'  {tag:24s}  no trades')
        return
    print(f'  {tag:24s}  {len(usd):>5,} trades   total {usd.sum():+10.0f} $   '
          f'mean {usd.mean():+7.1f} $   win {100*(usd > 0).mean():.1f}%')

print(f'\n════ Non-overlapping 1-BTC taker trades (hold {H_SEL} bars) ════')
for variant in ('ens', 'unan'):
    for j in (SEL_UP_J, SEL_DN_J):
        non_overlap(j, PRIMARY_RATE, variant)


In [ ]:
# ── Cell 11: GATE (pre-registered) ─────────────────────────────────────────
# At the h40 heads, primary rate 0.1%, ensemble variant, for at least one side:
#   A. pooled mean net bps > 0 AND day-bootstrap 95% CI excludes 0
#   B. pooled hit lift ≥ 3× AND per-fold lift ≥ 2× in ≥ 3 of 4 folds
#   C. neighbouring rates (0.01%, 1%) also pooled net > 0 (no magic threshold)
# Registered before seeing any test number (this cell was written with the
# notebook, 2026-07-08). Everything else in cell 10 is hypothesis-generating.
NEIGHBOUR_RATES = [r_ for r_ in TRIGGER_RATES if r_ != PRIMARY_RATE]

def gate_side(j):
    h, side = CLS_KEYS[j]
    pf, p = trigger_stats(j, PRIMARY_RATE, 'ens')
    A = bool(p['n'] > 0 and np.isfinite(p['net']) and p['net'] > 0
             and np.isfinite(p['ci'][0]) and p['ci'][0] > 0)
    ok_folds = sum(1 for r_ in pf if np.isfinite(r_['lift']) and r_['lift'] >= 2.0)
    B = bool(np.isfinite(p['lift']) and p['lift'] >= 3.0 and ok_folds >= N_FOLDS - 1)
    neigh = [trigger_stats(j, r_, 'ens')[1] for r_ in NEIGHBOUR_RATES]
    C = all(q['n'] > 0 and np.isfinite(q['net']) and q['net'] > 0 for q in neigh)
    print(f'  {side}{h} @ {PRIMARY_RATE*100:.1f}%:')
    print(f'    A  net {p["net"]:+.2f} bps, CI [{p["ci"][0]:+.2f}, {p["ci"][1]:+.2f}] '
          f'(need CI > 0)                → {"pass" if A else "fail"}')
    print(f'    B  lift {p["lift"]:.2f}x pooled (need ≥3), folds ≥2x: {ok_folds}/{N_FOLDS} '
          f'(need ≥{N_FOLDS-1})   → {"pass" if B else "fail"}')
    for r_, q in zip(NEIGHBOUR_RATES, neigh):
        print(f'    C  @{r_*100:6.2f}%: n={q["n"]:<5,} net {q["net"]:+.2f} bps')
    print(f'    C  neighbours net > 0                                  '
          f'→ {"pass" if C else "fail"}')
    return A and B and C

print('GATE (pre-registered):')
up_pass = gate_side(SEL_UP_J)
dn_pass = gate_side(SEL_DN_J)
GATE_PASS = up_pass or dn_pass
print(f'\n  → GATE {"PASSED" if GATE_PASS else "FAILED"}')
if GATE_PASS:
    feat_note = ('schema-3/4 features' if V3_AVAILABLE
                 else 'the v1 feature subset (v1-compat mode — not the schema-3/4 test)')
    print(f'  The selective idea works on {feat_note}. Next (run.007):')
    print('  threshold/θ tuning on val only, maker-entry + slippage simulation,')
    print('  and a freqtrade cross-check before believing any P&L number')
    print('  (run.005\'s EMA lesson: idealized sims flatter).')
else:
    print('  Read the failure mode before concluding anything:')
    print('  - lift ≈ 1 at every rate/head  → features still blind to')
    print('    opportunities; more schema-4 data first, then retest unchanged.')
    print('  - lift good but net ≤ 0        → θ/fee geometry wrong: the heads')
    print('    find moves but not enough beyond fees; revisit θ, h, TP-style exits.')
    print('  - CI straddles 0 with few triggers → not proven either way; rerun')
    print('    when the era is longer (this is the expected outcome of a short era).')


In [ ]:
# ── Cell 12: Plots ──────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
sweep = np.geomspace(3e-5, 3e-2, 12)

ax = axes[0, 0]
for j, color in ((SEL_UP_J, 'tab:green'), (SEL_DN_J, 'tab:red')):
    h, side = CLS_KEYS[j]
    for variant, ls in (('ens', '-'), ('reg', '--')):
        nets = [trigger_stats(j, r_, variant)[1]['net'] for r_ in sweep]
        ax.plot(sweep * 100, nets, ls, color=color,
                label=f'{side}{h} {"head" if variant == "ens" else "reg baseline"}')
ax.axhline(0, color='k', lw=0.5)
ax.axvline(PRIMARY_RATE * 100, color='gray', lw=0.5, ls=':')
ax.set_xscale('log'); ax.set_xlabel('nominal trigger rate (%)')
ax.set_ylabel('pooled net bps / signal'); ax.legend(fontsize=8)
ax.set_title('net-per-signal vs selectivity (dotted line = primary)')

ax = axes[0, 1]
for j, color in ((SEL_UP_J, 'tab:green'), (SEL_DN_J, 'tab:red')):
    h, side = CLS_KEYS[j]
    lifts = [trigger_stats(j, r_, 'ens')[1]['lift'] for r_ in sweep]
    ax.plot(sweep * 100, lifts, color=color, label=f'{side}{h}')
ax.axhline(1, color='k', lw=0.5)
ax.set_xscale('log'); ax.set_xlabel('nominal trigger rate (%)')
ax.set_ylabel('hit-rate lift vs base'); ax.legend(fontsize=8)
ax.set_title('does conviction actually concentrate hits?')

ax = axes[1, 0]
for j, color in ((SEL_UP_J, 'tab:green'), (SEL_DN_J, 'tab:red')):
    h, side = CLS_KEYS[j]
    p = prob_pool[:, j]; lab = L_all[ends_pool, j]
    edges = np.quantile(p, np.linspace(0, 1, 11))
    mids, obs = [], []
    for a, b in zip(edges[:-1], edges[1:]):
        m = (p >= a) & (p <= b)
        if m.sum() > 0:
            mids.append(p[m].mean()); obs.append(lab[m].mean())
    ax.plot(mids, obs, 'o-', color=color, label=f'{side}{h}')
lim = max(ax.get_xlim()[1], ax.get_ylim()[1])
ax.plot([0, lim], [0, lim], 'k--', lw=0.5, label='perfect calibration')
ax.set_xlabel('predicted P(opportunity)'); ax.set_ylabel('observed rate')
ax.legend(fontsize=8); ax.set_title(f'reliability, h{H_SEL} heads (prob deciles)')

ax = axes[1, 1]
step = max(len(ends_pool) // 5000, 1)
ax.plot(dt_all[ends_pool[::step]], close[ends_pool[::step]], color='0.7', lw=0.7)
for j, color, mk in ((SEL_UP_J, 'tab:green', '^'), (SEL_DN_J, 'tab:red', 'v')):
    E = []
    for r in results:
        tau = np.quantile(r['val_prob'][:, j], 1 - PRIMARY_RATE)
        E.append(r['test_ends'][r['test_prob'][:, j] >= tau])
    e = np.concatenate(E)
    ax.scatter(dt_all[e], close[e], s=12, color=color, marker=mk,
               label=f'{CLS_KEYS[j][1]}{CLS_KEYS[j][0]} trigger', zorder=3)
ax.legend(fontsize=8); ax.set_title(f'primary triggers ({PRIMARY_RATE*100:.1f}% nominal) on price')
ax.tick_params(axis='x', rotation=30)

plt.tight_layout()
plot_path = os.path.join(OUTPUT_DIR, 'run006_selective.png')
plt.savefig(plot_path, dpi=110, bbox_inches='tight')
plt.show()
print(f'Saved → {plot_path}')


In [ ]:
# ── Cell 13: Save to Drive + how to read this run ──────────────────────────
import shutil

model_path = os.path.join(OUTPUT_DIR, f'lstm_run006_h{H_SEL}.pt')
torch.save({
    'model_states': results[-1]['states'],     # all seeds, last (largest-train) fold
    'scaler':       results[-1]['scaler'],
    'features':     features,
    'horizons':     HORIZONS,
    'cls_keys':     CLS_KEYS,
    'seeds':        SEEDS,
    'config':       dict(seq_len=SEQ_LEN, hidden_dim=HIDDEN_DIM,
                         num_layers=NUM_LAYERS, dropout=DROPOUT,
                         vol_window=VOL_WINDOW, target_clip=TARGET_CLIP,
                         window_sec=WINDOW_SEC, h_trade=H_TRADE, h_sel=H_SEL,
                         theta_rel=THETA_REL, pos_w_cap=POS_W_CAP, bce_w=BCE_W,
                         sample_w_base=SAMPLE_W_BASE,
                         loss_w_alpha=LOSS_W_ALPHA, loss_w_cap=LOSS_W_CAP),
    'test_ic_per_fold':      [r['test_ic'] for r in results],
    'test_ap_per_fold':      [r['test_ap'] for r in results],
    # deployable thresholds: per-fold val quantiles at the primary rate
    'primary_tau_per_fold':  [{CLS_COLS[j]: float(np.quantile(r['val_prob'][:, j],
                                                              1 - PRIMARY_RATE))
                               for j in range(len(CLS_KEYS))} for r in results],
}, model_path)
print(f'Saved model → {model_path}')

from google.colab import drive
drive.mount('/content/drive')
os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)
for p in [scores_path, model_path, os.path.join(OUTPUT_DIR, 'run006_selective.png')]:
    if os.path.exists(p):
        shutil.copy(p, DRIVE_SAVE_DIR)
        print(f'  → Drive: {os.path.join(DRIVE_SAVE_DIR, os.path.basename(p))}')
drive.flush_and_unmount()

print('\nHow to read this run:')
print(' 1. Cell 5 first: era length, coverage, and the label base rates. If the')
print('    era is barely past MIN_ERA_DAYS, expect the gate to say "not proven".')
print(' 2. Cell 10 headline table: only the val-calibrated numbers are')
print('    deployable. hit% is the TP-order upper bound, net bps the hold-to-')
print('    horizon lower bound; truth in between. No slippage is modelled.')
print(' 3. Cell 11 GATE is the only pre-registered evidence (h40 heads, 0.1%,')
print('    ensemble). Exploratory tables are run.007 hypotheses, not results.')
print(' 4. The reg-baseline table is the control: if the heads do not beat it,')
print('    the BCE objective added nothing over the frozen regression score.')
print(' 5. Unanimity rows show whether seed disagreement is a usable "no idea"')
print('    detector — compare lift at equal realized /day, not equal nominal rate.')
print(' 6. If the gate passes: run.007 = θ/threshold tuning (val only), maker +')
print('    slippage sim, freqtrade cross-check. If it fails: read the failure-')
print('    mode guidance in cell 11 before changing anything.')
